In [ ]:
from callm.train_model import ModelTrainer

In [ ]:
    config = {
        "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "data_stream": "TRACE",
        "tasks": ["20Minuten"],
        "batch_size": 2,
        "num_epochs": 0.005,
        "learning_rate": 3e-4,
        "weight_decay": 0.01,
        "use_dora": False,
        "eval_step": 1,
        "save_step": 2,
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "optimizer": "adamw_8bit",
        "max_grad_norm": 0.3,
        "gradient_accumulation_steps": 4,
        "warmup_ratio": 0.01,
        "lr_scheduler_type": "linear",
        "lora_path": None,
        "lora_id": "lora_1"
    }

In [ ]:
#Initial training

trainer = ModelTrainer(config)
trainer.train_lora()

In [ ]:
#Follow up call to trainer with a new lora

#update config for new data (and potentially with new lora training config)
trainer.config['datasets'] = ['C-STANCE']
trainer.config['lora_id'] = 'lora_2'

#unload existing adapter from base model
trainer.model = trainer.model.unload()

#train new lora
trainer.train_lora()

In [ ]:
#Follow up call to trainer with an existing lora

#update config for new data (and potentially with new lora training config)
trainer.config['datasets'] = ['C-STANCE']
trainer.config['lora_path'] = 'lora_1'
trainer.config['lora_id'] = 'lora_1'

#unload existing adapter from base model
trainer.model = trainer.model.unload()

#continue to train existing lora
trainer.train_lora()

In [ ]:
#check if i can unload and load trained adapter - can't change name from 'default'
trainer.model = trainer.model.unload()
print(trainer.model.active_adapters)

trainer.load_existing_lora('lora_1')
print(trainer.model.active_adapters)

In [ ]:
# Check if unload gives base model - DONE
# Check if config can be updated with new data (and potentially more) - DONE
# Check if training existing lora works - DONE
# Check if i can retrieve name of active adapter - No, it is set to 'Default'

In [ ]:
#kill trainer and clean memory once it is done

import gc
import torch

def kill_trainer(trainer):
    """Kills the trainer object, unloads models, and cleans memory."""

    # Unload the models from memory
    if trainer.model is not None:
      trainer.model = trainer.model.unload()
      # Unload LoRA adapters if any
      # Replace "adapter_1" with the actual adapter name(s) if used
      # trainer.model.unload_adapter("adapter_1")

      del trainer.model  # Delete the model object

    # Delete the trainer object
    del trainer

    # Force garbage collection
    gc.collect()
    torch.cuda.empty_cache()

    print("Trainer object and associated models have been killed and memory cleaned.")